### ingest_daily_support_tickets

In [12]:
import sys
print(sys.executable)

c:\Users\prath\anaconda3\python.exe


In [2]:
! pip install boto3
! pip install python-dotenv
! pip install pymysql      #Mysql driver for python to connect to sql

import os            #For env variables
import pandas as pd     # For analysis
import boto3               #Its a SDK to interact with AWS services using python
from io import StringIO    #  Treat a normal string like a file.                  #
from sqlalchemy import create_engine      #For DB connection
from datetime import datetime, timedelta   #to deal with dates

from dotenv import load_dotenv
load_dotenv("sample.env")         #To load env file



# Example: Pandas read_csv() expects a file.

# But you have CSV data as a string (API response, logs, etc.)

# Without StringIO → Pandas will throw an error.
# With StringIO → Pandas thinks it's reading a real file.

# # df (your data)
# ↓
# convert df to CSV text
# ↓
# store CSV text in memory (StringIO)
# ↓
# upload that CSV text to S3 as an object

# os whenever Python code wants to access environment variables

# PyMySQL → only helps Python talk to MySQL (low-level)
# SQLAlchemy → helps Python work with databases in a clean, powerful way (high-level)


ERROR: Invalid requirement: '#Mysql': Expected package name at the start of dependency specifier
    #Mysql
    ^


True

In [3]:
# ---------- DB CONFIG ---------- Can get from DB under schema
db_config = {
    "host": "localhost",
    "port": "3306",
    "user": "root",  # change
    "password": "root", # change
    "database": "careplus_support_db"
}

# S3 configuration(Target bucket name)
S3_BUCKET = "care-plus-project-14062026" 
S3_PREFIX = "support-tickets-db/raw_tickets/"

DATE_TRACKER_FILE = "date_tracker.txt"    # to track last processed date,open() function takes a string file name and opens the file for you.

#AWS config from env(# Access keys become env, when yOU decide to store them as environment variables)

AWS_CONFIG = {
    "aws_access_key_id": os.getenv("AWS_ACCESS_KEY"),  
    "aws_secret_access_key": os.getenv("SECRET_KEY"),
    "region_name": os.getenv("REGION")
}

In [4]:
# ---------- UTILITY FUNCTIONS ----------

# Create db engine
# This is a SQLAlchemy connection URL.
# It tells SQLAlchemy:
# Which database you want to connect to
# Which driver to use
# What credentials to use
# Where the database is located

def get_engine(config):
    return create_engine(f"mysql+pymysql://{config['user']}:{config['password']}@{config['host']}:{config['port']}/{config['database']}")

def upload_to_s3(df, bucket, key):    #df is dataframe(data),target bucket, object 
    csv_buffer = StringIO()            #data is stored in memory not in disk to improve performance     
    df.to_csv(csv_buffer, index=False)  #conveting df to csv

    s3 = boto3.client('s3', **AWS_CONFIG)    #what service to use using config,generally use ** infront of config Take the dictionary named config
                                             #and unpack it into key=value arguments.
    s3.put_object(Bucket=bucket, Key=key, Body=csv_buffer.getvalue())    #uploading
    print(f"✅ Uploaded to s3://{bucket}/{key}")   

def read_last_date(file_path):       #last date 30 jun,read the date of july1st     
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            return f.read().strip()
    return "2025-06-30"  # Starting point before 1st July

def update_last_date(file_path, new_date):
    with open(file_path, 'w') as f:
        f.write(new_date)

def get_next_date(last_date_str):
    last_date = datetime.strptime(last_date_str, "%Y-%m-%d")
    next_date = last_date + timedelta(days=1)
    return next_date.strftime("%Y-%m-%d")

# ---------- MAIN INGESTION LOGIC ----------
def run_ingestion():
    engine = get_engine(db_config)
    last_date = read_last_date(DATE_TRACKER_FILE)
    next_date = get_next_date(last_date)

    # Query only that day’s data
    query = f"""
        SELECT * FROM support_tickets
        WHERE DATE(created_at) = '{next_date}';
    """
    df = pd.read_sql(query, engine)
    print(df.shape)
    print(df.head(10))

    if df.empty:
        print(f"⚠️ No data found for {next_date}. Skipping upload.")
        return

    # Upload to S3
    s3_key = f"{S3_PREFIX}support_tickets_{next_date}.csv"
    upload_to_s3(df, S3_BUCKET, s3_key)

    # Update date tracker
    update_last_date(DATE_TRACKER_FILE, next_date)
    print(f"📅 Updated tracker to {next_date}")

# Run
if __name__ == "__main__":
    run_ingestion()


# IMPORT happens when:
# ✔ Airflow loads your file
# ✔ Lambda loads your file
# ✔ Tests import your file
# ✔ Another script imports your file
# ✔ You split code into multiple files
# ✔ You reuse functions from this file

# Without __main__:
#     run_ingestion() runs automatically on import → BAD

# With __main__:
#     run_ingestion() runs ONLY when file is executed directly → GOOD

(23, 10)
    ticket_id        created_at       resolved_at  agent priority  \
0  TCK0710000  2025-07-10 00:04  2025-07-10 02:00  Arjun   Medium   
1  TCK0710001  2025-07-10 01:14  2025-07-11 08:40  Arjun   Medium   
2  TCK0710002  2025-07-10 01:26  2025-07-11 12:57  Arjun     High   
3  TCK0710003  2025-07-10 02:24  2025-07-10 23:53  Kavya      Low   
4  TCK0710004  2025-07-10 02:26  2025-07-10 23:39  Arjun    Medum   
5  TCK0710005  2025-07-10 02:30  2025-07-10 23:34  Kavya      Hgh   
6  TCK0710006  2025-07-10 03:00  2025-07-10 17:39  Arjun   Medium   
7  TCK0710007  2025-07-10 03:00  2025-07-10 12:08  Arjun    Medum   
8  TCK0710008  2025-07-10 03:14  2025-07-10 11:29  Kavya   Medium   
9  TCK0710009  2025-07-10 03:24  2025-07-10 15:52  Kavya      Low   

  num_interactions         IssUeCat   channel     status agent_feedback  
0                8       Bug Report      Chat   Resolved                 
1                5  Payment Failure  Web Form  Escalated                 
2        

In [5]:
# No need to run every time to upload every single data. So we are using looping

def run_ingestion_loop():
    while True:
        last_date = read_last_date(DATE_TRACKER_FILE)
        next_date = get_next_date(last_date)

        print(f"\n📅 Processing Date: {next_date}")

        run_ingestion()

        # Stop when no more data is available
        engine = get_engine(db_config)
        query = f"""
            SELECT * FROM support_tickets
            WHERE DATE(created_at) = '{next_date}';
        """
        df = pd.read_sql(query, engine)

        if df.empty:
            print("🚫 No more data found. Stopping pipeline.")
            break

if __name__ == "__main__":
    run_ingestion_loop()


📅 Processing Date: 2025-07-03
(26, 10)
    ticket_id        created_at       resolved_at   agent priority  \
0  TCK0703000  2025-07-03 00:25  2025-07-03 01:58   Arjun       Lw   
1  TCK0703001  2025-07-03 01:00  2025-07-03 16:32   Sneha   Medium   
2  TCK0703002  2025-07-03 01:19  2025-07-03 12:26   Sneha       Lw   
3  TCK0703003  2025-07-03 03:00              None   Arjun    Medum   
4  TCK0703004  2025-07-03 03:14  2025-07-03 16:08   Sneha   Medium   
5  TCK0703004  2025-07-03 03:14  2025-07-03 16:08   Sneha   Medium   
6  TCK0703005  2025-07-03 03:57  2025-07-03 17:32   Arjun   Medium   
7  TCK0703006  2025-07-03 04:39  2025-07-04 07:43  Ananya     High   
8  TCK0703007  2025-07-03 05:50  2025-07-03 16:29   Sneha     High   
9  TCK0703008  2025-07-03 05:55  2025-07-03 22:53  Ananya       Lw   

  num_interactions         IssUeCat   channel    status agent_feedback  
0                2  Feature Request      Chat  Resolved                 
1          -999999  Payment Failure  Web Fo

In [1]:
df.info()

NameError: name 'df' is not defined

In [1]:
#Processed Ticket parquet file

import pandas as pd

df=pd.read_parquet("C:/Users/prath/Downloads/run-1781683450387-part-block-0-r-00000-snappy.parquet")
df.head(135)

,ticket_id,created_at,resolved_at,agent,priority_cleaned,num_interactions,Issue_Category,channel,status
0,TCK0701000,2025-07-01 00:13:00,NaT,Sneha,Medium,7,Payment Failure,Chat,Open
1,TCK0701001,2025-07-01 01:24:00,2025-07-01 02:47:00,Kavya,Medium,8,Account Locked,Web Form,Resolved
2,TCK0701002,2025-07-01 01:49:00,2025-07-01 19:25:00,Rohit,Low,3,Bug Report,Web Form,Resolved
3,TCK0701004,2025-07-01 03:03:00,2025-07-02 11:00:00,Ananya,High,5,Bug Report,Chat,Resolved
4,TCK0701005,2025-07-01 03:07:00,2025-07-02 00:35:00,Arjun,High,1,Bug Report,Email,Resolved
...,...,...,...,...,...,...,...,...,...
130,TCK0706019,2025-07-06 09:21:00,2025-07-07 17:31:00,Sneha,Medium,5,Account Locked,Web Form,Resolved
131,TCK0706020,2025-07-06 10:08:00,2025-07-07 00:49:00,Arjun,Medium,6,Login Issue,Phone,Resolved
132,TCK0706020,2025-07-06 10:08:00,2025-07-07 00:49:00,Arjun,Medium,6,Login Issue,Phone,Resolved
133,TCK0706021,2025-07-06 10:18:00,2025-07-07 13:20:00,Rohit,Medium,5,Payment Failure,Email,Resolved


In [ ]:

#Curated logs

import pandas as pd

df=pd.read_parquet("C:/Users/prath/Downloads/support_logs_2025-07-07_curated.parquet")
df


,total_logs,error_count,warning_count,avg_cpu_usage,avg_response_time
0,62,4,0,0,985.693548


In [6]:
#Curated Tickets

import pandas as pd

df=pd.read_parquet("C:/Users/prath/Downloads/part-00035-651fef90-058f-460a-88ee-746023bf10c4-c000.snappy.parquet")
df


,total_tickets,resolved_tickets,open_tickets,high_priority,medium_priority,low_priority,avg_interactions,avg_resolution_hours,chat_tickets,email_tickets,phone_tickets,webform_tickets,payment_failure,account_locked,bug_report,login_issue
0,223,180,35,0,0,0,4.430493,17.202216,57,67,56,43,55,47,45,38
